# json extender

OK, so we've got everything operational. ish.

What we're going to do now is run the theories on the entire set of responses. Then we can update the json files so that they contain all the relevant information.

In [1]:
import amati_bea as amati

In [2]:
import pandas as pd

from sklearn.metrics import f1_score, cohen_kappa_score

In [3]:
from operator import itemgetter
import pickle
import datetime
import re
import glob
import json

## Get the data

We'll just be able to pull out the relevant data from the saved theory.

Let's find a trial response. First, need to pull out the question for that theory:

And then get the associated responses:

So let's just bag the first of those:

Reverse ferret, let's apply to the trial set:

In [4]:
def get_all_rule_applications(theory):
    "Don't restrict to trial"

    question_id=theory["question_id"]
    original_question_id=theory["original_question_id"]

    question_ss = theory['all_questions_df'].set_index("original_question_id").T[
        original_question_id
    ]
    
    trial_responses_df = theory['all_responses_df'].query("`question_id`==@question_id")

    trial_texts_used = [
        question_id,
        question_ss["correct_id"],
        question_ss["partially_correct_id"],
        question_ss["incorrect_id"],
    ] + list(trial_responses_df["response_id"])

    trial_text_df = theory['all_text_df'].query("`id`.isin(@trial_texts_used)")[
        ["id", "i", "lemma"]
    ]

    results_dict = amati.apply_theory(
        theory['theory'],
        responses_df=trial_responses_df,
        text_df=trial_text_df,
        rulesstring=theory['rules'],
        question_ss=question_ss,
    )

    """ compare_df = (
        trial_responses_df[["response_id", "score"]]
        .rename({"score": "actual"}, axis="columns")
        .set_index("response_id")
        .assign(
            prediction=results_df.ffill(axis="columns").iloc[:, -1].fillna("incorrect")
        )
    ) """

    return {"theory": theory, "applications": results_dict}

## Apply to all theories

OK, so now we should be able to add that information to each of the jsons in the actual files:

Because I changed the format of the identifiers, I'll need to wrangle it back into a form that uses the original ids.

First, let's get an id mapper. We can do it for all IDs. Start with the response IDs:

Then the question IDs:

OK, good. So now we want to get a representation of what rule applies to each response.

We want a function which returns a json form of the rule it's been given.

In [5]:
def get_json_from_rule(rule):
    if rule["selected_rule"][0] == "rule1":
        return {
            "amati": {
                "literals": [
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@1",
                    }
                ],
                "prediction": rule["predicted_grade"][0],
            }
        }
    elif rule["selected_rule"][0] == "rule2":
        return {
            "amati": {
                "literals": [
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@1",
                    },
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 2][0],
                        "id": "@2",
                    },
                    {"pred": "precedes", "order": ["@1", "@2"]},
                ],
                "prediction": rule["predicted_grade"][0],
            }
        }
    elif rule["selected_rule"][0] == "rule3":
        return {
            "amati": {
                "literals": [
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@1",
                    },
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 2][0],
                        "id": "@2",
                    },
                ],
                "prediction": rule["predicted_grade"][0],
            }
        }
    elif rule["selected_rule"][0] == "ki_rule1":
        return {
            "amati": {
                "literals": [
                    {
                        "pred": "contains",
                        "document": "student_response",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@1",
                    },
                    {
                        "pred": "contains",
                        "document": "reference_answer",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@2",
                    },
                    {
                        "pred": "contains",
                        "negated":True,
                        "document": "question",
                        "lemma": [x[1] for x in rule["parameters"] if x[0] == 1][0],
                        "id": "@3",
                    },
                ],
                "prediction": rule["predicted_grade"][0],
            }
        }
    else:
        assert(False)

OK, the code is now a complete mess, but I can tidy up at a future date. Key thing at the moment is to get things into a usable state whereby we can actually do a trial run.

In [6]:
def get_amati_json_from_theory(filename):

    # Pull the theory out of the file
    with open(filename, "rb") as fIn:
        theory = pickle.load(fIn)

    # Build an id mapper
    id_mapper = (
        theory["all_responses_df"]
        .set_index("response_id")["original_response_id"]
        .to_dict()
    )

    id_mapper.update(
        theory["all_questions_df"]
        .set_index("question_id")["original_question_id"]
        .to_dict()
    )

    # Apply the theory to all the responses to that question
    applications_dict = get_all_rule_applications(theory)["applications"]

    amati_pairs_ls = [
        (id, get_json_from_rule(rules[0]))
        for (id, rules) in applications_dict.items()
        if rules
    ] + [
        (id, {"amati": {"literals": {}, "prediction": "incorrect"}})
        for (id, rules) in applications_dict.items()
        if not rules
    ]

    # And convert to a correctly mapped dictionary
    amati_dict = {
        id_mapper[response_id]: amati for (response_id, amati) in amati_pairs_ls
    }

    return amati_dict

OK, well let's see if that works. I'll run it on all the theories, and check that we end up with a complete set of responses.

Obviously save the output as json before doing anything rash...

In [7]:
outputs_ls=[]

for f in glob.glob("theories/run_20260303/*"):
    print(f)
    try:
        outputs_ls.append(get_amati_json_from_theory(f))
    except:
        continue

theories/run_20260303/theory_10d57539-7664-4f0f-afe2-51a749036f54_2026-03-03_22:57:43.893904.pickle
theories/run_20260303/theory_badc70b3-e8b8-4d54-8b1b-4c25d9b371da_2026-03-03_22:58:33.553917.pickle
theories/run_20260303/theory_93b2ea65-14e0-4a3b-bb50-bb4759c44a22_2026-03-03_19:07:29.031321.pickle
theories/run_20260303/theory_2ad4c4b0-5dfe-4667-b344-a00434eeccf1_2026-03-04_02:38:16.725370.pickle
theories/run_20260303/theory_52af038f-7137-4406-bcbb-fb5a0291fd25_2026-03-03_17:24:32.763687.pickle
theories/run_20260303/theory_0bf8fd8c-a68a-4491-ae28-be767f335729_2026-03-03_17:58:04.395918.pickle
theories/run_20260303/theory_741376db-a5f7-43a1-abc4-988ff40cf642_2026-03-03_22:35:10.945536.pickle
theories/run_20260303/theory_b8d67d94-9aeb-4b96-a3d1-d7eebd6a1b9a_2026-03-03_23:00:31.130305.pickle
theories/run_20260303/theory_bcded41e-7008-4fb7-a64f-701f41dd2ed8_2026-03-03_20:16:47.950977.pickle
theories/run_20260303/theory_6cc84feb-e8e1-4561-8057-452dce98bd65_2026-03-03_19:41:37.701232.pickle


In [8]:
with open('test.json', 'w') as fOut:
    json.dump(outputs_ls, fOut)

In [10]:
d={}

for e in outputs_ls:
    d.update(e)

d

{'97fb8399-ecbc-4e6e-b9a4-fc6d0a3468b7': {'amati': {'literals': [{'pred': 'contains',
     'document': 'student_response',
     'lemma': '"gehen"',
     'id': '@1'}],
   'prediction': 'incorrect'}},
 'cda9ad99-4da6-4090-bd58-5339e096812a': {'amati': {'literals': [{'pred': 'contains',
     'document': 'student_response',
     'lemma': '"gehen"',
     'id': '@1'}],
   'prediction': 'incorrect'}},
 '0c6adcfe-852d-4c08-b7da-d08a90826772': {'amati': {'literals': [{'pred': 'contains',
     'document': 'student_response',
     'lemma': '"umgekehrt"',
     'id': '@1'}],
   'prediction': 'incorrect'}},
 'f34b9087-697e-47f1-9813-0c3d1dea19de': {'amati': {'literals': [{'pred': 'contains',
     'document': 'student_response',
     'lemma': '"graph"',
     'id': '@1'}],
   'prediction': 'incorrect'}},
 '67f3ffb6-8614-4a6f-aee4-3a9986764f4d': {'amati': {'literals': [{'pred': 'contains',
     'document': 'student_response',
     'lemma': '"umgekehrt"',
     'id': '@1'}],
   'prediction': 'incorrect'}

In [11]:
len(d)

7765

Now we can add things to the original sets.

In [ ]:
with open('/Users/alistair.willis/repos/bea-2026/BEA Shared Task 2026 on Rubric-based Short Answer Scoring for German/updated/3way/ALICE_LP_train_3way__v2.json') as fIn:
    json_train_3=json.load(fIn)
    
json_train_3[0]

In [21]:
json_train_3_amati=[]
for j in json_train_3:
    if j['id'] in d:
        j.update(d[j['id']])
    json_train_3_amati.append(j)


In [22]:
json_train_3_amati[0]

{'id': '2a5d7d84-fd00-4cb1-859c-246604d8e522',
 'question_id': 'bc2d01c9-424c-428a-9909-7114819a31d9',
 'question': 'Beurteile das Zerteilen der Äpfel hinsichlich der Geschwindigkeit des Verderblichkeitsprozesses (Hinweis: wenn Du fachliche Informationen zu chemischen Abläufen des Verderbens einbeziehen möchtest, kannst Du diese unter Angabe der Quellen recherchieren und einbinden).',
 'answer': 'Bei einem höheren Zerteilungsgrad ( hier das Zerteilen der Äpfel ) kommt es zu einer schnelleres Reaktionsgeschwindigkeit .\nDie hat man anhand unseres Modellversuches gesehen .\nAlso können wir feststellen , dass ein Stoff mit einem höheren Zerteilungsgrad einen Einfluss auf die Reaktionsgeschwindigkeit .\nSomit reagieren auch die zerschnittenden Äpfel schnelller .\n',
 'rubric': {'Incorrect': 'Die Schüler:innen beurteilen die Verderblichkeit der Äpfel nicht auf Basis des Kollisionsmodells',
  'Partially correct': 'Die Schüler:innen beurteilen die Verderblichkeit der Äpfel teilweise auf Basis

In [20]:
d['2a5d7d84-fd00-4cb1-859c-246604d8e522']

{'amati': {'literals': [{'pred': 'contains',
    'document': 'student_response',
    'lemma': '"einfluss"',
    'id': '@1'}],
  'prediction': 'incorrect'}}

In [23]:
with open('/Users/alistair.willis/repos/bea-2026/BEA Shared Task 2026 on Rubric-based Short Answer Scoring for German/updated/3way/ALICE_LP_trial_3way__v2.json') as fIn:
    json_trial_3=json.load(fIn)
    
json_trial_3[0]

{'id': '36646661-efef-4070-bcce-87273ca9a82b',
 'question_id': 'bc2d01c9-424c-428a-9909-7114819a31d9',
 'question': 'Beurteile das Zerteilen der Äpfel hinsichlich der Geschwindigkeit des Verderblichkeitsprozesses (Hinweis: wenn Du fachliche Informationen zu chemischen Abläufen des Verderbens einbeziehen möchtest, kannst Du diese unter Angabe der Quellen recherchieren und einbinden).',
 'answer': 'Durch Zerteilen der Äpfel ändert sich das Verhältnis von Volumen zur Oberfläche .\nWenn die Stücke kleiner sind , ist die Oberfläche im Verhältnis zum Volumen größer .\nDadurch reagieren wiederum mehr Teilchen .\nDie Äpfel sollten also nicht zerschnitten werden , wenn eine lange Haltbarkeit erreicht werden soll .\n',
 'rubric': {'Incorrect': 'Die Schüler:innen beurteilen die Verderblichkeit der Äpfel nicht auf Basis des Kollisionsmodells',
  'Partially correct': 'Die Schüler:innen beurteilen die Verderblichkeit der Äpfel teilweise auf Basis des Kollisionsmodells.',
  'Correct': 'Die Schüler:in

In [24]:
json_trial_3_amati=[]
for j in json_trial_3:
    if j['id'] in d:
        j.update(d[j['id']])
    json_trial_3_amati.append(j)


In [28]:
json_trial_3_amati[3]

{'id': '27bc7e48-c8d6-4c2d-a51d-9a28864eaab3',
 'question_id': 'bc2d01c9-424c-428a-9909-7114819a31d9',
 'question': 'Beurteile das Zerteilen der Äpfel hinsichlich der Geschwindigkeit des Verderblichkeitsprozesses (Hinweis: wenn Du fachliche Informationen zu chemischen Abläufen des Verderbens einbeziehen möchtest, kannst Du diese unter Angabe der Quellen recherchieren und einbinden).',
 'answer': 'äpfel verderben schneller bei höheren temperaturen , weil die reaktionsgeschwindigkeit angehoben wird\n',
 'rubric': {'Incorrect': 'Die Schüler:innen beurteilen die Verderblichkeit der Äpfel nicht auf Basis des Kollisionsmodells',
  'Partially correct': 'Die Schüler:innen beurteilen die Verderblichkeit der Äpfel teilweise auf Basis des Kollisionsmodells.',
  'Correct': 'Die Schüler:innen beurteilen die Verderblichkeit der Äpfel umfassend auf Basis des Kollisionsmodells.'},
 'score': 'Incorrect',
 'amati': {'literals': [{'pred': 'contains',
    'document': 'student_response',
    'lemma': '"äpf

In [62]:
with open('train_3way_with_prediction.json', 'w') as fOut:
    json.dump(json_train_3_amati, fOut)

with open('trial_3way_with_prediction.json', 'w') as fOut:
    json.dump(json_trial_3_amati, fOut)



In [57]:
df=pd.DataFrame([(j['id'],j['score']) for j in json_trial_3]).rename({0:'id', 1:'actual'}, axis='columns').set_index('id')

df['prediction']=pd.DataFrame([(j['id'],j['amati']['prediction']) for j in json_trial_3 if 'amati' in j]).set_index(0)

df['actual']=df['actual'].str.lower().str.replace("partially correct", 'partially_correct')

df

,actual,prediction
id,,
36646661-efef-4070-bcce-87273ca9a82b,incorrect,incorrect
962ea355-6849-46b5-ab50-c74e7c98da50,correct,incorrect
4f7f3884-963e-4c46-8458-a84c6014782e,correct,incorrect
27bc7e48-c8d6-4c2d-a51d-9a28864eaab3,incorrect,correct
8cd711d0-a0be-46af-832a-e9c30cc6ae03,correct,incorrect
...,...,...
50ebd5ae-eb16-4284-b22e-ae3169e42058,correct,incorrect
0f922c54-3c13-4683-b7a6-353588b79677,correct,correct
483689d8-0601-4e4f-9971-53803def8af6,incorrect,incorrect


In [60]:
df[df['prediction'].isna()]

,actual,prediction
id,,
725bffc4-89df-412a-a6fc-e7ac15e3d277,correct,NaN
83ab0d14-2a45-4fbb-a3d2-54a28b1b8969,partially_correct,NaN
08bd53b5-7ef5-4d08-aab0-ac14691f9ddf,correct,NaN
c03382c1-8dd6-4c50-8578-5282d8736153,partially_correct,NaN
ec64be94-04cf-47e2-be19-8a7e27c11f55,correct,NaN
8eca0fac-28ea-4c0f-aff9-624ae672fe8c,partially_correct,NaN
8774a3b2-6a6d-42ee-a69a-bc76f8c9153b,partially_correct,NaN
49817c87-8467-41ea-a653-bbe8da29867c,correct,NaN
c89445e1-d808-43c9-8306-c5ba973c9046,correct,NaN
